# 02. Preguntas de Investigación

Analiza las 9 preguntas con datos existentes.  
Criterio: clásico `< 2000`, moderno `>= 2000`.  
Q1: proxy por conteo de anuncios (clasificados).  
Q6/Q9: pendientes (requieren comercializadoras/entrevistas).

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.chdir('..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
df = pd.read_csv('data/processed/mercado_havana.csv', encoding='utf-8-sig')
print(len(df))

## Q1: Marca con mayor índice de publicaciones (proxy: anuncios)

In [ ]:
q1 = df['marca'].value_counts().head(10)
q1

## Q2: Promedio de precios por marca

In [ ]:
dfp = df[df['precio_usd'].notna() & (df['precio_usd']>0)].copy()
q2 = dfp.groupby('marca')['precio_usd'].mean().sort_values(ascending=False).head(10)
q2

## Q3: Relación precio-año

In [ ]:
dfa = dfp[dfp['anio']>0].copy()
if len(dfa)>0:
    print('r=', dfa[['anio','precio_usd']].corr().iloc[0,1])
    sns.scatterplot(data=dfa.sample(n=min(200,len(dfa)), random_state=42), x='anio', y='precio_usd'); plt.savefig('reports/figuras/q3_precio_vs_anio.png', bbox_inches='tight'); plt.show()
else: print('sin datos')

## Q5: Proporción clásicos (<2000) vs modernos (>=2000)

In [ ]:
dfa = df[df['anio']>0].copy()
if len(dfa)>0:
    dfa['tipo'] = dfa['anio'].apply(lambda a: 'clasico' if a < 2000 else 'moderno')
    print(dfa['tipo'].value_counts(normalize=True))
else: print('sin datos')

## Q7: Municipio con mayor índice de publicaciones

In [ ]:
q7 = df['municipio'].fillna('sin_municipio').value_counts().head(10)
q7

## Q8: Relación municipio-precio

In [ ]:
q8 = dfp.groupby('municipio')['precio_usd'].median().sort_values(ascending=False).head(10)
q8

## Q4: Marca con precios más elevados (top por mediana)

In [ ]:
q4 = dfp.groupby('marca')['precio_usd'].median().sort_values(ascending=False).head(10)
q4

## Q6 y Q9 (Pendientes)
- Q6 requiere `data/raw/comercializadoras/comercializadoras.csv` con precios (formal vs informal).  
- Q9 (costo de mantener un auto) requiere datos de las entrevistas en `data/raw/entrevistas/`.

## Q1: Marca con mayor índice de publicaciones (proxy: anuncios de Revolico)

Nota: Se trata de un *proxy* (conteo de anuncios publicados en clasificados de Revolico), no de redes sociales. Solo ámbito La Habana.

In [ ]:
q1 = df['marca'].fillna('no identificada').value_counts().head(15)
q1.to_csv('reports/tablas/q1_marca_mayor_indice_publicaciones.csv')
q1

## Q2: Promedio de precios por marca (mediana preferida por robustez ante outliers)

In [ ]:
q2_med = dfp.groupby('marca')['precio_usd'].median().sort_values(ascending=False).head(15)
q2_med.to_csv('reports/tablas/q2_mediana_precio_por_marca.csv')
q2_med

## Q4: Marca con los precios más elevados (mediana por marca)

In [ ]:
q4 = dfp.groupby('marca')['precio_usd'].median().sort_values(ascending=False).head(15)
q4.to_csv('reports/tablas/q4_marca_precios_mas_elevados_mediana.csv')
q4

## Q3: Relación entre precio y año de fabricación

In [ ]:
dfa = dfp[dfp['anio']>0].copy()
import numpy as np
if len(dfa)>0:
    corr = dfa[['anio','precio_usd']].corr().iloc[0,1]
    print(f'Correlación Pearson (anio, precio_usd): {corr:.4f}')
    # Regresión lineal simple
    x = dfa['anio']
    y = dfa['precio_usd']
    m,b = np.polyfit(x,y,1)
    print(f'Regresión: precio_usd = {m:.2f}*anio + {b:.2f} (aprox {m:.1f} USD/año)')
    plt.figure(figsize=(8,6))
    sns.scatterplot(data=dfa.sample(n=min(300,len(dfa)), random_state=42), x='anio', y='precio_usd', alpha=0.5)
    plt.plot(x, m*x+b, color='#111827', linewidth=2, label=f'y={m:.1f}x+{b:.1f}')
    plt.title('Relación entre año de fabricación y precio (La Habana)')
    plt.xlabel('Año')
    plt.ylabel('Precio (USD)')
    plt.legend()
    plt.savefig('reports/figuras/q3_precio_vs_anio_tendencia.png', bbox_inches='tight', dpi=150)
    plt.show()
else:
    print('sin datos')

## Q5: Proporción de anuncios de autos clásicos vs modernos

Criterio: clásico `< 2000`, moderno `>= 2000`

In [ ]:
dfa = df[df['anio']>0].copy()
if len(dfa)>0:
    dfa['tipo'] = dfa['anio'].apply(lambda a: 'clasico' if a < 2000 else 'moderno')
    prop = dfa['tipo'].value_counts(normalize=True)
    cnt = dfa['tipo'].value_counts()
    res = pd.DataFrame({'conteo': cnt, 'proporcion': prop})
    res.to_csv('reports/tablas/q5_clasicos_vs_modernos.csv')
    print(res)
    plt.figure(figsize=(6,6))
    plt.pie(res['conteo'], labels=res.index, autopct='%1.1f%%', colors=['#6B7280','#2563EB'])
    plt.title('Proporción clásicos vs modernos (La Habana)')
    plt.savefig('reports/figuras/q5_clasicos_vs_modernos.png', bbox_inches='tight', dpi=150)
    plt.show()
else:
    print('sin datos')

## Q7: Municipio con mayor índice de publicaciones

In [ ]:
q7 = df['municipio'].fillna('sin_municipio').value_counts()
q7.to_csv('reports/tablas/q7_municipio_mayor_indice_publicaciones.csv')
q7.head(15)

## Q8: Relación entre municipio y precio de venta (mediana por municipio)

In [ ]:
q8 = dfp.groupby('municipio')['precio_usd'].agg(['count','median','mean']).sort_values('median', ascending=False)
q8.to_csv('reports/tablas/q8_municipio_precio_mediana.csv')
q8.head(15)

## Q6: Diferencia entre precios del mercado formal y el mercado informal

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Cargar formal (comercializadoras)
formal = pd.read_csv('data/processed/comercializadoras.csv', encoding='utf-8-sig')
# Solo vehículos
formal_v = formal[formal['tipo_item']=='vehiculo'].copy()
formal_v = formal_v[formal_v['precio_usd'].notna() & (formal_v['precio_usd']>0)].copy()

# Informal La Habana (activos con precio)
informal = df[df['precio_usd'].notna() & (df['precio_usd']>0)].copy()
# Filtrar activos si existe estado_verificacion
if 'estado_verificacion' in informal.columns:
    informal_act = informal[informal['estado_verificacion']=='activo'].copy()
else:
    informal_act = informal.copy()

print('Formal (vehículos):', len(formal_v))
print('Informal La Habana (activos con precio):', len(informal_act))

# Comparación general
comp = pd.DataFrame({
  'mercado': ['formal','informal'],
  'n': [len(formal_v), len(informal_act)],
  'mediana_usd': [formal_v['precio_usd'].median(), informal_act['precio_usd'].median()],
  'media_usd': [formal_v['precio_usd'].mean(), informal_act['precio_usd'].mean()],
  'p25': [formal_v['precio_usd'].quantile(0.25), informal_act['precio_usd'].quantile(0.25)],
  'p75': [formal_v['precio_usd'].quantile(0.75), informal_act['precio_usd'].quantile(0.75)],
})
comp.to_csv('reports/tablas/q6_formal_vs_informal_comparacion.csv', index=False)
comp

In [ ]:
# Visualización comparativa
plt.figure(figsize=(8,6))
data = [formal_v['precio_usd'], informal_act['precio_usd']]
bp = plt.boxplot(data, showfliers=False)
plt.xticks([1,2], ['Formal (comercializadoras)', 'Informal La Habana (Revolico)'])
plt.title('Comparación de precios: mercado formal vs informal')
plt.ylabel('Precio (USD)')
plt.savefig('reports/figuras/q6_formal_vs_informal_boxplot.png', bbox_inches='tight', dpi=150)
plt.show()

# Por marca donde hay cruce
formal_m = formal_v.groupby('marca_norm')['precio_usd'].median().rename('mediana_formal')
informal_m = informal_act.groupby('marca')['precio_usd'].median().rename('mediana_informal')
# normalizar marcas informales
inf_norm = informal_act.copy()
inf_norm['marca_norm'] = inf_norm['marca'].str.lower()
informal_mn = inf_norm.groupby('marca_norm')['precio_usd'].median().rename('mediana_informal')
cruce = pd.concat([formal_m, informal_mn], axis=1).dropna()
cruce['diferencia_formal_informal'] = cruce['mediana_formal'] - cruce['mediana_informal']
cruce = cruce.sort_values('mediana_formal', ascending=False)
cruce.to_csv('reports/tablas/q6_cruce_por_marca.csv')
cruce.head(15)

## Q9: ¿Cuál es el costo de mantener un auto en la actualidad?

Requiere datos de entrevistas en `data/raw/entrevistas/Audios/`. Se inventariarán audios y se dejará plantilla estructurada; sin transcripciones disponibles no se infieren cifras numéricas.

In [ ]:
import os, json
from pathlib import Path

audios = list(Path('data/raw/entrevistas/Audios').glob('*'))
print('Audios encontrados:', len(audios))
for a in sorted(audios):
    print('-', a.name, a.stat().st_size, 'bytes')

plantilla = {
  'fuente': 'entrevistas',
  'requerido': ['combustible_mensual_cup_usd', 'repuestos_mensual', 'mantenimiento_preventivo_anual', 'seguro_tramites', 'otros', 'moneda'],
  'notas': 'No se inventan valores. Completar tras transcripción/validación.'
}
with open('reports/tablas/q9_plantilla_costos_mantenimiento.json','w',encoding='utf-8') as f:
    json.dump(plantilla,f,ensure_ascii=False,indent=2)
print('Plantilla guardada en reports/tablas/q9_plantilla_costos_mantenimiento.json')